## Init

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/axl.dxn@gmail.com/atlikon_pipeline/1_setup/utilities

## Configure widgets

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(f"catalog: {catalog}, data_source: {data_source}")

## Read from gross price bronze table 

In [0]:
df_bronze = (
    spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
)

display(df_bronze.limit(10))

## Normalize `month` field

In [0]:
df_bronze.select('month').distinct().show()

In [0]:
# Parse 'month' from multiple possible formats
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = (
    df_bronze
    .withColumn(
        "month",
        F.coalesce(
            F.try_to_date(F.col("month"), "yyyy/MM/dd"),
            F.try_to_date(F.col("month"), "dd/MM/yyyy"),
            F.try_to_date(F.col("month"), "yyyy-MM-dd"),
            F.try_to_date(F.col("month"), "dd-MM-yyyy")
        )
    )
)

In [0]:
df_silver.select('month').distinct().show()

## Handling 'gross_price'

In [0]:
# Converting only valid numeric values to double
# Fixing negative prices by making them positive
# Replacing all non-numeric values with 0

df_silver = (
    df_silver
    .withColumn(
        "gross_price",
        F.when(
            F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'), 
            F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double"))
             .otherwise(F.col("gross_price").cast("double"))
        ).otherwise(0)
    )
)

display(df_silver.limit(10))

## Enrich silver dataset 
By performing an inner join with the products table to fetch the correct product_code for each product_id

In [0]:
df_products = spark.table("fmcg.silver.products") 
df_joined = (
    df_silver
    .join(
        df_products.select("product_id", "product_code"), 
        on="product_id", 
        how="inner"
    )
)
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")

display(df_joined.limit(10))

## Write in silver table 

In [0]:
(
    df_joined
    .write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("mergeSchema", "true")
    .mode("overwrite") 
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")
)

## Sanity check of silver table

In [0]:
query = f"SELECT * FROM {catalog}.{silver_schema}.{data_source} LIMIT 10;"

df_check = spark.sql(query)

display(df_check.limit(10))